# Dependencies (for venv)

In [1]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


# Generate the dataset

In [2]:
import random
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')   
# device = torch.device('cpu')
torch.set_default_device(device)

print(torch.get_default_device())

cuda:0


In [3]:
def generate_objects(N=3, block_size=0.2, initial_tower=True):
    # 1. Generate random order for the goals
    idx_goals = [i for i in range(0, N)]
    random.shuffle(idx_goals)

    # 2. Generate initial order for the blocks by inverting the order of goals
    idx_objs = idx_goals[::-1]

    # 3. Generate goals
    # https://www.generationrobots.com/media/panda-franka-emika-datasheet.pdf
    # according to the datasheet, the Franka Panda can reach in an area of 0.855 [m] around itself
    max_reach_dist = 0.7 # [m]

    # select a random (x,y) position for the goal.
    x_goal = random.random() * max_reach_dist
    y_goal = random.random() * max_reach_dist

    pos_goals = [(x_goal, y_goal, i*block_size) for i in range(0, N)]

    # Generate initial positions
    if initial_tower:
        x_obj = random.random() * max_reach_dist
        y_obj = random.random() * max_reach_dist

        pos_objects = [(x_obj, y_obj, i*block_size) for i in range(0, N)]
    else:
        pos_objects = [(random.random() * max_reach_dist, random.random() * max_reach_dist, i*block_size) for i in range(0, N)]

    goals = {}
    objects = {}
    for i in range(0, N):
        goals[idx_goals[i]] = pos_goals[i]
        objects[idx_objs[i]] = pos_objects[i]

    # print(f"goals (in desired order of final stacking):\n{goals}")
    # print(f"initial object positions (in order of stacking):\n{objects}")

    return objects, goals, idx_objs, idx_goals


In [4]:
# Now solve the stacking problem by taking the highest block in the objects tower (last in order)
# and putting it into the lowest available goal (first in order)

# build the state vector, ordered nodes with categorical, position and binary features, followed by ordered goals with the same features
def build_state(objects, goals):
    s = []
    N = len(objects)
    for n in range(0, N):
        # one hot encode of blocks (all nodes are either blocks or goals, all objects are goals)
        s.append(1)
        s.append(0)

        # one hot encode of whether block is in goal
        s.append(int(objects[n] == goals[n]))
        s.append(int(objects[n] != goals[n]))

        # position of block
        for o in objects[n]: s.append(o)

    for n in range(0, N):
        # one hot encode of goals (all nodes are either blocks or goals, all objects are goals)
        s.append(0)
        s.append(1)

        # one hot encode of whether block is in goal (goal if filled)
        s.append(int(objects[n] == goals[n]))
        s.append(int(objects[n] != goals[n])) # there was a wrong inequality sign here!

        # position of block
        for o in goals[n]: s.append(o)

    return s

def build_example(N=3, block_size=0.2, initial_tower=True):
    objects, goals, idx_objs, idx_goals = generate_objects(N, block_size, initial_tower)
    
    tau = []    
    tau.append(build_state(objects, goals))

    for n in range(0, N):
        # print(f"step #{n}")
        
        o = N-1-n

        # print(f"choosing object #{idx_objs[o]} with pos {objects[idx_objs[o]]}")
        # print(f"moving object #{idx_objs[o]} to goal{goals[idx_objs[o]]}")

        objects[idx_objs[o]] = goals[idx_objs[o]]
        tau.append([o, o])

        # print(f"Updated object positions with goals:")
        # print(f"goals:\n{goals}")
        # print(f"current objects pos:\n{objects}")
        # print("\n")

        tau.append(build_state(objects, goals))

    # TODO (?): add the empty action to a complete graph
    # print(tau)
    return tau

In [5]:
# Alternatevely
# at each step, find the highest block that is not yet in its goal
# and move it to the lowest available goal
# repeat until each blocks is in its goal

# Observation to graph

In [6]:
# Now put the demonstration in graph format, using pytorch geometric

import torch
import torch.nn.functional as F
import torch_geometric
from torch_geometric.utils.convert import to_networkx
import torch_geometric.transforms as T
import networkx as nx

def build_dataset(N=3, N_EXAMPLES=50):
    data_list = []
    for i in range(N_EXAMPLES):
        print(f"Building example #{i}")
        tau = build_example(N)

        states = tau[0:-1:2] # exclude the complete graph
        actions = tau[1::2]

        # print(len(states), states)
        # print(len(states), actions)

        # build edges, only once
        # graphs are always fully connected, undirected, and with no edge features and no self loops
        edges = torch.tensor([(i,j) for i in range(N*2) for j in range(N*2) if i != j]).t().contiguous()
        transform = T.Cartesian(norm=False, cat=False)
        # build nodes and labels
        for n in range(0, N):
            s = states[n]
            a = actions[n]

            # torch_geometric.data.Data is a single graph
            # x is the input graph, as a matrix of dimension [node, node_features]
            # I have N*2 nodes (N blocks, N goals) and 5 features each
            x = torch.reshape(torch.tensor(s), (N*2, -1)) # N*2 considers both goals and blocks
            
            # additive measurement noise
            # z = torch.zeros(x.shape)
            # z[:, 2:] = torch.rand((z[:, 2:].shape))*0.005
            # x += z

            # y is divided into two vectors, one hot encoded, probabilities for the block and the goal
            # outputs are probabilities for each index to match dimensions with the mask
            y = torch.Tensor.float(F.one_hot(torch.tensor(a), num_classes=N))
            # y = (y[:, 0].reshape(-1), y[:, 1].reshape(-1))
            # print(y)
            y = torch.flatten(y)

            # y = torch.tensor(a)
            
            # keeping track of pos is useful to apply to transform and generate edge_attr later
            pos = x[:, 4:]  
            # x = x[:, :4]

            object_mask = x[:, 0] == 1 # for minibatch splitting during training
            # print(object_mask)
            # y = torch.tensor(a)
            print(x)
            G = torch_geometric.data.Data(x=x, edge_index=edges, pos=pos, y=y, object_mask=object_mask)
            # embed the distance between the objects as an edge feature (in edge_attr)
            # Transform.Cartesian derives it directly from G.pos
            G = transform(G)

            # save all the graphs into a single data_list for use as a DataLoader later
            data_list.append(G)

        # print(data_list)
    return data_list

/home/emamaker/university/Università/magistrale/anno-2/semestre-1/nn/project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/emamaker/university/Università/magistrale/anno-2/semestre-1/nn/project/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


# Build torch_geometric dataset from examples

In [9]:
# https://pytorch-geometric.readthedocs.io/en/latest/tutorial/create_dataset.html#frequently-asked-questions
# just create a DataLoader starting from the list of example Data
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

# train_dataset = build_dataset(N=3, N_EXAMPLES=100)
train_dataset = build_dataset(N=4, N_EXAMPLES=80)# + build_dataset(N=4, N_EXAMPLES=50)
# print("Training dataset: ", train_dataset)

train_loader = DataLoader(train_dataset, batch_size=8)
train2_loader = DataLoader(train_dataset, batch_size=1)
val_loader = DataLoader(build_dataset(N=3, N_EXAMPLES=10), batch_size=1)
# val_loader = DataLoader(build_dataset(N=3, N_EXAMPLES=10) + build_dataset(N=4, N_EXAMPLES=10), batch_size=1, shuffle=True)
test_loader = DataLoader(build_dataset(N=3, N_EXAMPLES=10), batch_size=1)

Building example #0
tensor([[1.0000, 0.0000, 0.0000, 1.0000, 0.1429, 0.0578, 0.6000],
        [1.0000, 0.0000, 0.0000, 1.0000, 0.1429, 0.0578, 0.0000],
        [1.0000, 0.0000, 0.0000, 1.0000, 0.1429, 0.0578, 0.2000],
        [1.0000, 0.0000, 0.0000, 1.0000, 0.1429, 0.0578, 0.4000],
        [0.0000, 1.0000, 0.0000, 1.0000, 0.0051, 0.2255, 0.0000],
        [0.0000, 1.0000, 0.0000, 1.0000, 0.0051, 0.2255, 0.6000],
        [0.0000, 1.0000, 0.0000, 1.0000, 0.0051, 0.2255, 0.4000],
        [0.0000, 1.0000, 0.0000, 1.0000, 0.0051, 0.2255, 0.2000]],
       device='cuda:0')
tensor([[1.0000, 0.0000, 1.0000, 0.0000, 0.0051, 0.2255, 0.0000],
        [1.0000, 0.0000, 0.0000, 1.0000, 0.1429, 0.0578, 0.0000],
        [1.0000, 0.0000, 0.0000, 1.0000, 0.1429, 0.0578, 0.2000],
        [1.0000, 0.0000, 0.0000, 1.0000, 0.1429, 0.0578, 0.4000],
        [0.0000, 1.0000, 1.0000, 0.0000, 0.0051, 0.2255, 0.0000],
        [0.0000, 1.0000, 0.0000, 1.0000, 0.0051, 0.2255, 0.6000],
        [0.0000, 1.0000, 0.0000

# Build NN

In [10]:
# https://pytorch-geometric.readthedocs.io/en/latest/modules/nn.html
# https://pytorch-geometric.readthedocs.io/en/latest/get_started/introduction.html
# https://colab.research.google.com/github/Alireza-Akhavan/graph-neural-network/blob/main/07-Graph_Classification_pyg.ipynb#scrollTo=CN3sRVuaQ88l
# https://khairulislam.github.io/GNN/pages/graph/Graph_Classification.html

# regarding the use of position (and generalization to unseen positions): https://github.com/pyg-team/pytorch_geometric/issues/3326
# regarding fully connected graphs and the use of dense layers: https://github.com/pyg-team/pytorch_geometric/issues/964

from torch.nn import Linear
from torch_geometric.nn import DenseSAGEConv, DenseGATConv
from torch_geometric.utils import to_dense_adj, to_dense_batch, to_nested_tensor

import lightning as L
from lightning.pytorch.callbacks import Callback, EarlyStopping
from lightning.pytorch.callbacks.early_stopping import EarlyStoppingReason

class GCN(torch.nn.Module):
    def __init__(self, hidden_dim=64,dropout=0.5):
        super().__init__()
        # All policies consist of 3 hidden layers, with 64 hidden units each and ReLU activation.
        # For attention policies, the number of attention heads were set to 1.
        self.conv1 = DenseSAGEConv(7, hidden_dim, normalize=True)
        self.conv2 = DenseSAGEConv(hidden_dim, hidden_dim, normalize=True)
        self.conv3 = DenseSAGEConv(hidden_dim, hidden_dim, normalize=True)
        # self.conv1 = DenseGATConv(5, hidden_nodes, dropout=dropout)
        # self.conv2 = DenseGATConv(hidden_nodes, hidden_nodes, dropout=dropout)
        # self.conv3 = DenseGATConv(hidden_nodes, hidden_nodes, dropout=dropout)
        # self.lin1 = Linear(hidden_nodes, hidden_nodes, bias=True)
        self.lin2 = Linear(hidden_dim, 1, bias=True)
        self.dropout = dropout

    def forward(self, data):
        x, edge_index, batch, object_mask = data.x, data.edge_index, data.batch, data.object_mask # batch is the index of the example

        # some noise. Probably wrong to be added here but simpler than in batch
        
        # print(data.num_nodes)
        # print("x", x, x.shape)
        # print("edge_index", edge_index)
        # print("batch", batch)

        # x, b = to_dense_batch(x, batch, max_num_nodes=1)
        # print("Dense batch")
        # print(x, b)
        # print(x.shape, b.shape)

        # additive measurement noise during training
        # if self.training:
            # z = torch.zeros(x.shape)
            # z[:, 2:] = torch.rand((z[:, 2:].shape))*0.005
            # x += z

        # print(object_mask, object_mask.shape)

        x, _ = to_dense_batch(x, batch)
        mask, _ = to_dense_batch(object_mask, batch)
        adj = to_dense_adj(edge_index, batch)
        # print(x, x.shape)
        # print(mask, mask.shape)
        # print(adj, adj.shape)

        x = self.conv1(x, adj)
        x = F.relu(x)
        # print(x, x.shape)
        # x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, adj)
        x = F.relu(x)
        # x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv3(x, adj)
        x = F.relu(x)
        # x = F.dropout(x, p=self.dropout, training=self.training)
        # x = self.lin1(x)
        # x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)

        # print(x, x.shape)

        x = x.flatten()

        # # The output of the GNN policy is K + L dimensional, reshaped into K and L dimensional outputs
        # mask_dense = mask.reshape(x.shape)
        # print(mask, mask.shape)

        objects_logits = x[data.object_mask]
        objects_batch = batch[data.object_mask]


        goals_logits = x[data.object_mask]
        goals_batch = batch[data.object_mask]

        # since these outputs are interpreted as unnormalized logits, the fill value should be a large negative number to discourage the choice
        objects_dense, objects_mask = to_dense_batch(objects_logits, objects_batch, fill_value=-1e3)
        goals_dense, goals_mask = to_dense_batch(goals_logits, goals_batch, fill_value=-1e3)

        # print(objects_dense, objects_dense.shape)
        # print(goals_dense, goals_dense.shape)

        return objects_dense, goals_dense, objects_mask, goals_mask


model = GCN(hidden_dim=32, dropout=0.1).to(device)
model.train()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-6)


tacc_obj = []
tacc_goal = []
vacc_obj = []
vacc_goal = []

for epoch in range(3000):
    model.train()
    Loss = torch.zeros((1))
    
    for i, batch in enumerate(train_loader):
        optimizer.zero_grad()
        objects_dense, goals_dense, objects_mask, goals_mask = model(batch)

        # print(objects_dense)

        y_obj_dense, _ = to_dense_batch(batch.y[batch.object_mask], batch.batch[batch.object_mask], fill_value=-1)
        y_obj_dense = y_obj_dense.reshape(objects_dense.shape)

        y_goal_dense, _ = to_dense_batch(batch.y[~batch.object_mask], batch.batch[~batch.object_mask], fill_value=-1)
        y_goal_dense = y_goal_dense.reshape(goals_dense.shape)

        # print(objects_dense, objects_dense.shape, y_obj_dense, y_obj_dense.shape)
        
        l1 = F.cross_entropy(objects_dense, y_obj_dense, ignore_index=-1)
        # loss = l1
        l2 = F.cross_entropy(goals_dense, y_goal_dense, ignore_index=-1)
        loss = l1+l2
        Loss += loss * batch.num_graphs

        loss.backward()
        optimizer.step()

    print(f'Training loss (CE): {Loss.item()}')


    def accuracy(loader):
        correct_obj = 0.
        correct_goal = 0.
        dist = {0: 0, 1: 0, 2: 0, 3:0 }

        for i,batch in enumerate(loader):
            objects_dense, goals_dense, objects_mask, goals_mask = model(batch)

            y_obj_dense, _ = to_dense_batch(batch.y[batch.object_mask], batch.batch[batch.object_mask], fill_value=-1)
            y_obj_dense = y_obj_dense.reshape(objects_dense.shape)

            y_goal_dense, _ = to_dense_batch(batch.y[~batch.object_mask], batch.batch[~batch.object_mask], fill_value=-1)
            y_goal_dense = y_goal_dense.reshape(goals_dense.shape)

            # print(objects_dense, objects_dense.shape)
            # print(batch.y)
            # print(y_obj_dense, y_obj_dense.shape)
            # print(torch.argmax(objects_dense)==batch.y[0], torch.argmax(objects_dense), batch.y[0])

            # take the argmax for the nodes in each batch
            correct_obj += (torch.argmax(objects_dense, dim=1) == torch.argmax(y_obj_dense, dim=1)).sum().item()
            correct_goal += (torch.argmax(goals_dense, dim=1) == torch.argmax(y_goal_dense, dim=1)).sum().item()
            dist[torch.argmax(objects_dense, dim=1).item()] += 1

        # print(correct_obj, correct_goal)
        acc_obj = correct_obj / i
        acc_goal = correct_goal / i

        return acc_obj, acc_goal, dist

    model.eval()
    
    acc_obj, acc_goal, dist = accuracy(train2_loader)
    print(f'Test Accuracy (objects/goals): {acc_obj:.4f}/{acc_goal:.4f}')
    # print("Distribution of predictions: ", dist)
    tacc_obj.append(acc_obj)
    tacc_goal.append(acc_goal)

    
    acc_obj, acc_goal, dist = accuracy(val_loader)
    print(f'Validation Accuracy (objects/goals): {acc_obj:.4f}/{acc_goal:.4f}')
    # print("Distribution of predictions: ", dist)
    vacc_obj.append(acc_obj)
    vacc_goal.append(acc_goal)

    # break

Training loss (CE): 888.4332885742188
Test Accuracy (objects/goals): 0.2602/0.2602
Validation Accuracy (objects/goals): 0.4138/0.4138
Training loss (CE): 886.5181274414062
Test Accuracy (objects/goals): 0.2727/0.2727
Validation Accuracy (objects/goals): 0.4828/0.4828
Training loss (CE): 887.8690185546875
Test Accuracy (objects/goals): 0.2602/0.2602
Validation Accuracy (objects/goals): 0.3793/0.3793
Training loss (CE): 887.0272216796875
Test Accuracy (objects/goals): 0.2759/0.2759
Validation Accuracy (objects/goals): 0.5517/0.5517
Training loss (CE): 887.1060180664062
Test Accuracy (objects/goals): 0.2790/0.2790
Validation Accuracy (objects/goals): 0.5517/0.5517
Training loss (CE): 887.7445678710938
Test Accuracy (objects/goals): 0.2696/0.2696
Validation Accuracy (objects/goals): 0.5172/0.5172
Training loss (CE): 886.9614868164062
Test Accuracy (objects/goals): 0.2696/0.2696
Validation Accuracy (objects/goals): 0.5172/0.5172
Training loss (CE): 886.6795043945312
Test Accuracy (objects/g

KeyboardInterrupt: 